# 50. Run LOSO K-Sweep Model Selection

This public-facing notebook runs the Stage-5 broad LOSO model-order sweep for the fusion HMM.

## Manuscript linkage
- Main Methods 2.5.1
- Supplementary Methods 1.6
- Figure 2 support
- Table S8 support

## What this notebook does
- runs the preserved broad K-sweep over `K = 2..12`
- writes held-out free-energy summaries and feasibility diagnostics
- records the 1-SE rule and the local-minimum screening outputs

## Important interpretation note
This notebook is the screening stage only. Higher-K solutions, including `K=12`, can still remain visible after screening. The final manuscript choice of `K=3` was not treated as coming from one fully automatic rule alone. It combined these screening outputs with the later shortlist stability and interpretability check.

## Inputs expected
- Stage-4 final segment branch under the canonical no-lag `minlen15` dataset
- `segments_manifest.tsv` and the segment `.npy` files it references

## Outputs written
- `cv_results.tsv`
- `cv_candidates_long.tsv`
- `fold_meta.tsv`
- `summary_byK_selected.tsv`
- `summary_byK_candidates.tsv`
- `paired_tests_vs_bestK.tsv`
- `K_selection_recommendation.json`
- free-energy and feasibility plots

## Environment note
The preserved source notebook is GPU-sensitive and was developed around TensorFlow, `osl_dynamics`, XLA-off execution, and chunk/resume behavior under long runs. CPU-only execution is possible in code but may be much slower in practice.


In [ ]:
from pathlib import Path

# -------------------------
# User-editable paths
# -------------------------
STAGE4_FINAL_ROOT = Path("<SET_STAGE4_FINAL_ROOT>")
STAGE5_OUTPUT_ROOT = Path("<SET_STAGE5_OUTPUT_ROOT>")

# -------------------------
# Canonical public dataset branch
# -------------------------
DATA_VARIANT = "intermediate"
FEATURE_MODE = "nolags"
MINLEN = 15

# Optional explicit manifest override
# Leave as None to use the standard Stage-4 location.
MANIFEST_TSV = None

# Screening grid
K_GRID = list(range(2, 13))

# Chunk/resume and debug controls
MAX_NEW_PAIRS_PER_RUN = 15
DEBUG_MAX_FOLDS = None
DEBUG_K_GRID = None
DEBUG_SEEDS = None

# GPU memory cap used by the preserved notebook. Set to None for memory-growth mode.
GPU_MEMORY_LIMIT_MB = 4096

FINAL_ROOT = STAGE4_FINAL_ROOT / DATA_VARIANT
OUT_ROOT = STAGE5_OUTPUT_ROOT / f"{DATA_VARIANT}_{FEATURE_MODE}_minlen{MINLEN}"
SOURCE_NOTEBOOK = Path("R01_PipelineC2_Ksweep_LOSO_OOMFIX_steps_XLAoff_chunkresume_v2REBATCH.ipynb")

print("FINAL_ROOT:", FINAL_ROOT)
print("OUT_ROOT:", OUT_ROOT)
print("SOURCE_NOTEBOOK:", SOURCE_NOTEBOOK)


In [ ]:
from stage5_hmm_selection_helpers import run_wrapped_k_sweep_source

run_wrapped_k_sweep_source(
    source_notebook=SOURCE_NOTEBOOK,
    final_root=FINAL_ROOT,
    out_root=OUT_ROOT,
    data_variant=DATA_VARIANT,
    feature_mode=FEATURE_MODE,
    minlen=MINLEN,
    manifest_tsv=MANIFEST_TSV,
    k_grid=K_GRID,
    max_new_pairs_per_run=MAX_NEW_PAIRS_PER_RUN,
    gpu_memory_limit_mb=GPU_MEMORY_LIMIT_MB,
    debug_max_folds=DEBUG_MAX_FOLDS,
    debug_k_grid=DEBUG_K_GRID,
    debug_seeds=DEBUG_SEEDS,
)


## Notes

- This notebook intentionally keeps the broad screening stage separate from the shortlist stability check in `51_run_loso_shortlist_stability_checks.ipynb`.
- The public default stays on the canonical Stage-4 branch: `FEATURE_MODE = "nolags"`, `MINLEN = 15`.
- Older lagged and `minlen10` provenance paths are preserved in archive notebooks but are not the public default here.
